In [1]:
# Imports
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
from diffrax import *
from controls import *

In [5]:
# (1) Dynamics of stoch vol. We want to solve this one without any correlation/lead-lag. Therefore we just use an Ito solver.

gamma1 = 1
gamma2 = 1 # note this is not Lipschitz and could explode...
sigma1 = 1.5
sigma2 = 1.5
alpha = 1.5
beta = 1.5

t0, t1 = 0, 1
epsilon = 0.01
lag = epsilon # = 1.2*epsilon was not really necessary
solver_epsilon = 0.1*epsilon # = 0.01*epsilon is really necessary in these examples
times = jnp.arange(t0, t1, epsilon)
num_samples = 1000
dim_bm = 2

y0 = jnp.array([1, 1])
drift_jones = lambda t, y, args: jnp.array([0, alpha + beta*y[1]])
diffusion_jones = lambda t, y, args: jnp.array([[jnp.sqrt(y[1])*y[0], 0], [sigma1*jnp.power(y[1], gamma1), sigma2*jnp.power(y[1], gamma2)]])
saveat = SaveAt(ts = times)

key=jr.PRNGKey(512)
split_key = jax.random.split(key, num_samples)

solutions = vmap_batch_solve(split_key, epsilon, dim_bm, drift_jones, diffusion_jones, y0, Midpoint(), t0, t1, solver_epsilon, saveat)

In [4]:
# Let's try and see whether we can gain in speed on a CPU with pmap

num_samples = 1000
split_key = jax.random.split(key, num_samples)

pmap_batch_solve = jax.vmap(batch_solve, in_axes=(0, None, None, None, None, None, None, None, None, None, None))

solutions = pmap_batch_solve(split_key, epsilon, dim_bm, drift_jones, diffusion_jones, y0, Midpoint(), t0, t1, solver_epsilon, saveat)

# no substantial improvement over vmap

In [12]:
import jax
import jax.numpy as jnp
from jax import vmap, pmap
from timeit import default_timer as timer
from concurrent.futures import ProcessPoolExecutor
import numpy as np

# Define a simple JAX function
def jax_function(x):
    return jnp.sin(x) ** 2

# Prepare input data
x = jnp.linspace(0, 10, 100000000)

# vmap
start = timer()
vmap_result = vmap(jax_function)(x)
end = timer()
print(f"vmap time: {end - start} seconds")

# pmap - ensure you have enough devices for this to make sense
# For CPU, you might need to set XLA_FLAGS environment variable
# export XLA_FLAGS="--xla_force_host_platform_device_count=<num_cores>"
x_pmap = jnp.stack([x for _ in range(jax.local_device_count())])
start = timer()
pmap_result = pmap(jax_function)(x_pmap)
end = timer()
print(f"pmap time: {end - start} seconds")

# concurrent.futures
from parallel_functions import *

# Convert JAX array to NumPy for use with concurrent.futures
x_np = np.linspace(0, 10, 100000000)

def parallel_numpy_function(inputs):
    with ProcessPoolExecutor() as executor:
        result = list(executor.map(numpy_function, np.array_split(inputs, executor._max_workers)))
    return np.concatenate(result)

start = timer()
futures_result = parallel_numpy_function(x_np)
end = timer()
print(f"concurrent.futures time: {end - start} seconds")

# vmap wins on my laptop, pmap a close second, concurrent.futures is by far the slowest

vmap time: 0.28372700000181794 seconds
pmap time: 0.27726308300043456 seconds
concurrent.futures time: 6.516381041001296 seconds


In [1]:
import os
os.cpu_count()

8